# Existing Model GPU Prediction on Colab (`0.2.1`)

This notebook validates prediction on a GPU-enabled Colab runtime with an already extracted model directory.

Use it when your model is already available in `/content` or Google Drive, for example after extracting a RepOD archive.

It runs:
- repository clone
- dependency install
- CUDA runtime check
- GPU prediction with an existing model
- preview of `predictions.txt` and `predictions.txt.map`

Notes:
- GPU is mainly useful for `TransformerJobOffersClassifier`
- the existing compatibility pytest currently covers CPU, so this notebook uses the direct CLI `predict` path on GPU


In [ ]:
%cd /content
!rm -rf /content/job-ads-classifier
!git clone --branch v0.2.1 https://github.com/OJALAB/job-ads-classifier.git /content/job-ads-classifier
%cd /content/job-ads-classifier
!python -m pip install --upgrade pip
!pip install -r requirements-colab-0.2.0.txt


In [ ]:
import subprocess

import torch

subprocess.run(["nvidia-smi"], check=False)
print("torch:", torch.__version__)
print("cuda_available:", torch.cuda.is_available())
print("device_count:", torch.cuda.device_count())
if torch.cuda.is_available():
    print("device_name:", torch.cuda.get_device_name(0))
assert torch.cuda.is_available(), (
    "This notebook expects a GPU runtime. In Colab choose Runtime > Change runtime type > GPU."
)


## Configure your existing model path

If the model lives in Google Drive, uncomment the mount lines in the next cell and point `MODEL_DIR` to the extracted model directory.


In [ ]:
from pathlib import Path
import subprocess

def run_command(command):
    print("$", " ".join(command))
    subprocess.run(command, check=True)

# Optional Google Drive mount.
# from google.colab import drive
# drive.mount("/content/drive")

MODEL_CLASSIFIER = "TransformerJobOffersClassifier"
MODEL_DIR = "/content/existing-model"
INPUT_PATH = "/content/job-ads-classifier/tests/data/x_test.txt"
OUTPUT_PATH = "/content/predictions-existing-gpu.txt"
PRECISION = "16"
THREADS = "1"

assert MODEL_CLASSIFIER in {"LinearJobOffersClassifier", "TransformerJobOffersClassifier"}
assert Path(MODEL_DIR).exists(), f"Missing model dir: {MODEL_DIR}"
assert Path(INPUT_PATH).exists(), f"Missing input file: {INPUT_PATH}"

print("MODEL_CLASSIFIER =", MODEL_CLASSIFIER)
print("MODEL_DIR =", MODEL_DIR)
print("INPUT_PATH =", INPUT_PATH)
print("OUTPUT_PATH =", OUTPUT_PATH)


In [ ]:
command = [
    "python",
    "main.py",
    "predict",
    MODEL_CLASSIFIER,
    "-x",
    INPUT_PATH,
    "-m",
    MODEL_DIR,
    "-p",
    OUTPUT_PATH,
    "-T",
    THREADS,
]

if MODEL_CLASSIFIER == "TransformerJobOffersClassifier":
    command.extend(["-A", "gpu", "-P", PRECISION])

run_command(command)


In [ ]:
run_command(["head", "-n", "5", OUTPUT_PATH])
run_command(["head", "-n", "5", f"{OUTPUT_PATH}.map"])
